In [0]:
from pyspark.sql import SparkSession

In [0]:
spark = SparkSession.builder.appName("leetcode-sql-7").getOrCreate

In [0]:
from pyspark.sql.types import StructType,StructField,StringType,IntegerType

sales_schema = StructType([
    StructField("sale_id", IntegerType(), False),
    StructField("product_id", IntegerType(), False),
    StructField("year", IntegerType(), False),
    StructField("quantity", IntegerType(), True),
    StructField("price", IntegerType(), True)
])

product_schema = StructType([
    StructField("product_id", IntegerType(), False),
    StructField("product_name", StringType(), True)
])

# 3. Create Input Data including Edge Cases
sales_data = [
    # Standard Example Rows
    (1, 100, 2008, 10, 5000),
    (2, 100, 2009, 12, 5000),
    (7, 200, 2011, 15, 9000),
    
    # Edge Cases:
    (3, 100, 2010, 0, 0),        # Zero quantity and zero price
    (4, 999, 2022, 5, 1200),     # Foreign key missing in Product table (product_id 999)
    (5, 200, 2012, 1, 9000),     # Second sale for same product_id (different year/sale_id)
]

product_data = [
    # Standard Example Rows
    (100, "Nokia"),
    (200, "Apple"),
    (300, "Samsung"),            # Edge Case: Product with no corresponding sales
    
    # Edge Cases:
    (400, "LG"),                 # Edge Case: Another product with no sales
    (500, None)                  # Edge Case: Product name is NULL
]

sales_df = spark.createDataFrame(sales_data,schema=sales_schema)
product_df = spark.createDataFrame(product_data,schema=product_schema)

In [0]:
joined_data = product_df.join(sales_df, on="product_id",how="inner")
joined_data.show()

+----------+------------+-------+----+--------+-----+
|product_id|product_name|sale_id|year|quantity|price|
+----------+------------+-------+----+--------+-----+
|       100|       Nokia|      1|2008|      10| 5000|
|       100|       Nokia|      2|2009|      12| 5000|
|       200|       Apple|      7|2011|      15| 9000|
|       100|       Nokia|      3|2010|       0|    0|
|       200|       Apple|      5|2012|       1| 9000|
+----------+------------+-------+----+--------+-----+



In [0]:
from pyspark.sql.functions import col
joined_data.filter(col("price")>0).select(col("product_name"),col("year"),col("price")).show()

+------------+----+-----+
|product_name|year|price|
+------------+----+-----+
|       Nokia|2009| 5000|
|       Nokia|2008| 5000|
|       Apple|2012| 9000|
|       Apple|2011| 9000|
+------------+----+-----+

